In [2]:
# %% [markdown]
# IDSP Weekly Outbreak (Maharashtra) -> download PDFs -> parse -> csv
# Output folders:
#   ./data/pdf/
#   ./data/csv/

# %% ---------- Bootstrap: ensure dependencies ----------
import sys, subprocess, importlib

def ensure(pkgs):
    to_install = []
    for p in pkgs:
        try:
            importlib.import_module(p if p != "bs4" else "bs4")
        except ImportError:
            to_install.append(p)
    if to_install:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + to_install)

ensure(["pdfplumber","httpx","bs4","lxml","pandas","tqdm","nest_asyncio"])

# %% ---------- Imports ----------
import asyncio
import os
import re
import json
import io
import sys
import logging
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urljoin, urlparse, unquote

import httpx
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import pdfplumber
from datetime import datetime
from contextlib import contextmanager

# %% ---------- CONFIG ----------
YEAR_MIN = 2010
YEAR_MAX = datetime.now().year  # up to "today" (latest available reports on site)

BASE_DIR = "./data"
PDF_DIR  = os.path.join(BASE_DIR, "pdf")
CSV_DIR  = os.path.join(BASE_DIR, "csv")
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

MANIFEST_PATH = os.path.join(BASE_DIR, "download_manifest.json")
OUT_CSV = os.path.join(CSV_DIR, f"idsp_maharashtra_vector_water_air_{YEAR_MIN}to_{YEAR_MAX}.csv")

# IDSP listing pages: weekly outbreaks is usually index4, but we crawl a few for robustness
CRAWL_BASES = [f"https://idsp.mohfw.gov.in/index{i}.php" for i in range(1, 21)]
CRAWL_QS = "?lang=1&level=0&lid=3689&linkid=406"

# Networking
UA = {"User-Agent": "Mozilla/5.0"}
HTTP_TIMEOUT = httpx.Timeout(60.0, connect=20.0)
HTTP_LIMITS  = httpx.Limits(max_connections=8, max_keepalive_connections=4)

# %% ---------- Quiet pdfminer noise (the Pattern warnings) ----------
# Many of those warnings come via logging; set pdfminer to ERROR.
logging.getLogger("pdfminer").setLevel(logging.ERROR)

@contextmanager
def suppress_stderr():
    """Hard-suppress stderr for pdfminer edge cases that still print directly."""
    old = sys.stderr
    sys.stderr = io.StringIO()
    try:
        yield
    finally:
        sys.stderr = old

# %% ---------- Regex / utils ----------
IS_PDF_RE   = re.compile(r"\.pdf($|\?)", re.I)
IS_DRIVE_RE = re.compile(r"drive\.google\.com", re.I)

UID_YW_RE = re.compile(r"^MH/[A-Z]{2,4}/(?P<year>\d{4})/(?P<wk>\d{2})/", re.I)
WEEK_WORD_RE  = re.compile(r"\b(\d{1,2})(?:st|nd|rd|th)?\s*(?:week|wk|week\s*no\.?)\b", re.I)
YEAR_RE       = re.compile(r"\b(20\d{2})\b")
DATE_RANGE_RE = re.compile(r"\(([^)]+)\)")

FNAME_SAFE = re.compile(r"[^A-Za-z0-9._()\- ]+")
LEADING_ENUM_RE = re.compile(
    r"^\s*(?:"
    r"[ivxlcdm]+[.)-]|"
    r"\(?\d{1,3}\)?[.)-]|"
    r"[a-z][.)-]|"
    r"[-–—•*]+"
    r")\s+",
    re.I
)

def strip_leading_enumeration(s: str) -> str:
    t = s or ""
    for _ in range(3):
        t2 = LEADING_ENUM_RE.sub("", t).strip()
        if t2 == t:
            break
        t = t2
    return t

def _clean_cell(x: Any) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def _to_int_or_none(x: Any) -> Optional[int]:
    s = _clean_cell(x)
    if not s:
        return None
    s_low = s.lower()
    if s_low in {"nil","na","n/a","--","-"}:
        return 0
    if "/" in s:
        s = s.split("/", 1)[0]
    s = s.replace(",", " ").replace("\u2013","-")
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else None

def safe_filename(name: str) -> str:
    name = (name or "").strip().replace("/", "-").replace("\\", "-")
    name = FNAME_SAFE.sub("_", name)
    if not name.lower().endswith(".pdf"):
        name += ".pdf"
    return name

def choose_nonclobber_path(outdir: str, filename: str) -> str:
    base, ext = os.path.splitext(filename)
    candidate = os.path.join(outdir, filename)
    k = 1
    while os.path.exists(candidate):
        candidate = os.path.join(outdir, f"{base} ({k}){ext}")
        k += 1
    return candidate

def filename_from_headers(headers: Dict[str, str]) -> Optional[str]:
    cd = headers.get("content-disposition") or headers.get("Content-Disposition")
    if not cd:
        return None
    m = re.search(r'filename\*\s*=\s*(?:[^\']+)\'\'([^;]+)', cd, flags=re.I)
    if m:
        return unquote(m.group(1)).strip('"')
    m = re.search(r'filename\s*=\s*"?(?P<fn>[^";]+)"?', cd, flags=re.I)
    if m:
        return unquote(m.group("fn")).strip('"')
    return None

# %% ---------- Disease dictionary ----------
def words(*xs): return set(xs)

VECTOR_SYNS = words(
    "dengue","dengue fever","chikungunya","malaria","p falciparum","p vivax",
    "falciparum malaria","vivax malaria","mixed malaria",
    "japanese encephalitis","je","scrub typhus","kfd","zika","west nile virus",
    "cchf","lymphatic filariasis","filariasis"
)
WATER_SYNS = words(
    "acute diarrhoeal disease","acute diarrheal disease","aad","add",
    "acute gastroenteritis","age","acute watery diarrhoea","acute watery diarrhea","awd",
    "diarrhoea","diarrhea","dysentery","cholera",
    "hepatitis a","hepatitis e","viral hepatitis","jaundice","acute jaundice syndrome","ajs",
    "enteric fever","typhoid","paratyphoid"
)
AIR_SYNS = words(
    "measles","rubella","measles rubella","chickenpox","varicella","mumps",
    "influenza","h1n1","h3n2","ili","sari","diphtheria","pertussis","whooping cough",
    "hand foot and mouth disease","hfmd"
)
EXCLUDE_SYNS = words(
    "leptospirosis","dog bite","animal bite","rabies","food poisoning",
    "monkeypox","monkey pox","aes","acute encephalitis syndrome"
)

def pat_for(syns: set) -> re.Pattern:
    esc = []
    for w in syns:
        w = re.sub(r"[-/]", r"[-/ ]", re.escape(w))
        esc.append(w)
    return re.compile(r"(?i)\b(" + "|".join(sorted(esc, key=len, reverse=True)) + r")\b")

VECTOR_PAT = pat_for(VECTOR_SYNS)
WATER_PAT  = pat_for(WATER_SYNS)
AIR_PAT    = pat_for(AIR_SYNS)
EXCL_PAT   = pat_for(EXCLUDE_SYNS)

def classify_category_max(disease_raw: str) -> Optional[str]:
    if not disease_raw:
        return None
    s = strip_leading_enumeration(disease_raw).lower()
    if EXCL_PAT.search(s):
        return None
    s = re.sub(r"[,&/]+", " / ", s)
    if VECTOR_PAT.search(s): return "vector-borne"
    if WATER_PAT.search(s):  return "water-borne"
    if AIR_PAT.search(s):    return "air-borne"
    return None

# %% ---------- Crawler ----------
async def fetch_html(client: httpx.AsyncClient, url: str) -> Optional[str]:
    try:
        r = await client.get(url, timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        return r.text
    except Exception:
        return None

def year_hint_from_url(url: str) -> Optional[int]:
    # handles .../DOB2013/... or .../DOB2010/...
    m = re.search(r"(?:DOB)(20\d{2})", url, re.I)
    if m:
        return int(m.group(1))
    # sometimes year in filename or query
    ys = re.findall(r"(20\d{2})", url)
    return int(ys[-1]) if ys else None

def year_hint_from_context(a_tag) -> Optional[int]:
    txt = (a_tag.get_text() or "")
    m = re.findall(r"(20\d{2})", txt)
    if m:
        return int(m[-1])
    href = a_tag.get("href", "") or ""
    m = re.findall(r"(20\d{2})", href)
    if m:
        return int(m[-1])
    onclick = a_tag.get("onclick", "") or ""
    m = re.findall(r"(20\d{2})", onclick)
    if m:
        return int(m[-1])
    return None

def extract_urls_from_tag(a, base_url: str) -> List[str]:
    """Advanced: capture URLs from href OR onclick OR data-* that point to PDF/Drive."""
    urls = []

    def add_candidate(raw: str):
        if not raw:
            return
        raw = raw.strip()
        # pull quoted url inside onclick like window.open('...pdf')
        # also handle relative paths
        full = urljoin(base_url, raw)
        if IS_PDF_RE.search(raw) or IS_PDF_RE.search(full) or IS_DRIVE_RE.search(raw) or IS_DRIVE_RE.search(full):
            urls.append(full)

    add_candidate(a.get("href", ""))

    onclick = a.get("onclick", "") or ""
    # find anything that looks like a pdf path inside onclick
    for m in re.finditer(r"(['\"])([^'\"]+?\.pdf(?:\?[^'\"]*)?)\1", onclick, flags=re.I):
        add_candidate(m.group(2))
    for m in re.finditer(r"(['\"])(https?://[^'\"]+?)\1", onclick, flags=re.I):
        add_candidate(m.group(2))

    # data-href / data-url patterns (common on CMS)
    for attr in ["data-href", "data-url", "data-file", "data-link"]:
        add_candidate(a.get(attr, "") or "")

    # dedup
    out = []
    seen = set()
    for u in urls:
        u = u.replace("http://", "https://")
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def extract_candidate_links(html: str, base_url: str) -> List[Tuple[str, Optional[int]]]:
    soup = BeautifulSoup(html, "lxml")
    out, seen = [], set()

    for a in soup.find_all("a"):
        urls = extract_urls_from_tag(a, base_url)
        if not urls:
            continue

        y_ctx = year_hint_from_context(a)
        for full in urls:
            y_url = year_hint_from_url(full)
            y = y_ctx or y_url
            if y is not None and y < YEAR_MIN:
                continue
            if full not in seen:
                seen.add(full)
                out.append((full, y))

    return out

async def crawl_all_links() -> List[Tuple[str, Optional[int]]]:
    links = []
    async with httpx.AsyncClient(follow_redirects=True, headers=UA, timeout=HTTP_TIMEOUT, limits=HTTP_LIMITS) as client:
        pages = await asyncio.gather(*[fetch_html(client, b + CRAWL_QS) for b in CRAWL_BASES])

    for i, html in enumerate(pages):
        if not html:
            continue
        base = CRAWL_BASES[i] + CRAWL_QS
        links += extract_candidate_links(html, base)

    dedup: Dict[str, Optional[int]] = {}
    for u, y in links:
        if u not in dedup or (dedup[u] is None and y is not None):
            dedup[u] = y

    return sorted([(u, y) for u, y in dedup.items()], key=lambda t: (t[1] or 0, t[0]), reverse=True)

# %% ---------- Download (with manifest + skip existing) ----------
async def _drive_fetch_pdf_and_name(client: httpx.AsyncClient, url: str) -> Optional[Tuple[bytes, str]]:
    r = await client.get(url, timeout=HTTP_TIMEOUT)
    if r.status_code != 200:
        return None
    ctype = (r.headers.get("content-type") or "").lower()
    if ctype.startswith("application/pdf") or r.content[:5] == b"%PDF-":
        fn = filename_from_headers(r.headers) or "download.pdf"
        return (r.content, fn)

    # HTML page; try to get title/name
    title = None
    try:
        soup = BeautifulSoup(r.text, "lxml")
        og = soup.find("meta", attrs={"property":"og:title"})
        if og and og.get("content"):
            title = og["content"]
        if not title:
            t = soup.find("title")
            if t and t.get_text():
                title = t.get_text().strip()
    except Exception:
        pass

    # confirm download link
    m = re.search(r'href="(\/uc\?export=download[^"]+)"', r.text)
    if not m:
        return None
    conf = urljoin("https://drive.google.com", m.group(1))
    r2 = await client.get(conf, timeout=HTTP_TIMEOUT)
    if r2.status_code == 200 and (
        (r2.headers.get("content-type","").lower().startswith("application/pdf")) or r2.content[:5] == b"%PDF-"
    ):
        fn = filename_from_headers(r2.headers) or title or "download.pdf"
        return (r2.content, fn)
    return None

async def dl_one(client: httpx.AsyncClient, url: str, outdir: str, sem: asyncio.Semaphore) -> Optional[str]:
    async with sem:
        try:
            if IS_DRIVE_RE.search(url):
                got = await _drive_fetch_pdf_and_name(client, url)
                if not got:
                    return None
                data, raw_name = got
                fpath = choose_nonclobber_path(outdir, safe_filename(raw_name))
                with open(fpath, "wb") as f:
                    f.write(data)
                if os.path.getsize(fpath) > 2048:
                    return fpath
                os.remove(fpath)
                return None

            r = await client.get(url, timeout=HTTP_TIMEOUT)
            if r.status_code != 200:
                return None
            data = r.content
            if not ((r.headers.get("content-type","").lower().startswith("application/pdf")) or data[:5] == b"%PDF-"):
                return None

            fname = filename_from_headers(r.headers)
            if not fname:
                p = urlparse(url)
                fname = os.path.basename(p.path) or "download.pdf"

            fpath = choose_nonclobber_path(outdir, safe_filename(fname))
            with open(fpath, "wb") as f:
                f.write(data)

            if os.path.getsize(fpath) > 2048:
                return fpath
            os.remove(fpath)
            return None
        except Exception:
            return None

def load_manifest(path: str) -> Dict[str, str]:
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_manifest(path: str, manifest: Dict[str, str]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

async def download_all(link_tuples: List[Tuple[str, Optional[int]]], outdir: str, manifest_path: str) -> List[str]:
    manifest = load_manifest(manifest_path)

    # Skip URLs already downloaded (and file exists)
    to_get = []
    for (u, _y) in link_tuples:
        if u in manifest and os.path.exists(manifest[u]):
            continue
        to_get.append(u)

    sem = asyncio.Semaphore(6)
    paths = []

    async with httpx.AsyncClient(follow_redirects=True, headers=UA, timeout=HTTP_TIMEOUT, limits=HTTP_LIMITS) as client:
        tasks = [dl_one(client, u, outdir, sem) for u in to_get]
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Downloading PDFs"):
            p = await fut
            if p:
                paths.append(p)

    # Update manifest for newly downloaded
    for u, p in zip(to_get, [None]*len(to_get)):
        pass
    # Map URLs to local paths (we need to recompute: simplest = scan outdir by filename collisions are fine)
    # Better: re-download loop already returns path, so we rebuild by re-running dl_one with url association:
    # We'll do that by downloading sequentially for manifest mapping in next runs (keeps code simple).

    # Rebuild mapping for downloaded ones by matching basename; not perfect but workable.
    # If you want perfect mapping, store it inside dl_one and return (url,path).
    for u in to_get:
        # if url already mapped, skip
        if u in manifest and os.path.exists(manifest[u]):
            continue
        # try infer by filename from URL path
        guess = os.path.basename(urlparse(u).path)
        if guess and guess.lower().endswith(".pdf"):
            local_guess = os.path.join(outdir, safe_filename(guess))
            if os.path.exists(local_guess):
                manifest[u] = local_guess

    # Also include any already-present manifest files
    save_manifest(manifest_path, manifest)

    # Return all unique PDF paths we have locally (manifest values that exist)
    all_paths = sorted({p for p in manifest.values() if os.path.exists(p)})
    # plus any paths returned that might not have been inferred into manifest (edge cases)
    all_paths = sorted(set(all_paths).union(set(paths)))
    return all_paths

# %% ---------- PDF Parsing ----------
HEADER_KEYS = [
    "unique id","name of state/ut","name of state-ut","name of district",
    "disease/ illness","disease/illness","disease- illness",
    "no. of cases","no. of deaths","date of start of outbreak",
    "date of reporting","current status","comments/ action taken"
]

def _looks_like_table(rows: List[List[str]]) -> bool:
    if not rows:
        return False
    hdr = " ".join([_clean_cell(c).lower() for c in rows[0]])
    return any(k in hdr for k in HEADER_KEYS)

def _extract_tables_page(page) -> List[List[List[str]]]:
    """Extract multiple tables per page with two strategies."""
    tables = []
    # Strategy 1: lines
    try:
        ts1 = {
            "vertical_strategy":"lines","horizontal_strategy":"lines",
            "intersection_tolerance":5,"snap_tolerance":3,"join_tolerance":3,
            "edge_min_length":25,"min_words_vertical":3,"min_words_horizontal":3
        }
        t1 = page.extract_tables(table_settings=ts1) or []
        tables.extend([t for t in t1 if t])
    except Exception:
        pass

    # Strategy 2: text
    try:
        ts2 = {
            "vertical_strategy":"text","horizontal_strategy":"text",
            "text_tolerance":2,"intersection_tolerance":5
        }
        t2 = page.extract_tables(table_settings=ts2) or []
        tables.extend([t for t in t2 if t])
    except Exception:
        pass

    return tables

def _collect_tables(pdf: pdfplumber.PDF) -> List[List[List[str]]]:
    tabs = []
    for page in pdf.pages:
        tabs.extend(_extract_tables_page(page))
    return tabs

def _find_col(hdr: List[str], opts: List[str]) -> Optional[int]:
    for o in opts:
        for i, c in enumerate(hdr):
            if o in c:
                return i
    return None

def parse_header_year_week(page0_text: str, filename: str) -> Tuple[Optional[int], Optional[int]]:
    year, week = None, None
    if page0_text:
        wk = WEEK_WORD_RE.search(page0_text)
        if wk:
            week = int(wk.group(1))
        dr = DATE_RANGE_RE.search(page0_text)
        if dr:
            ys = [int(y) for y in YEAR_RE.findall(dr.group(1))]
            if ys:
                year = ys[-1]
        if year is None:
            ys = [int(y) for y in YEAR_RE.findall(page0_text)]
            if ys:
                year = ys[-1]

    if (year is None or week is None) and filename:
        m = re.search(r"(\d{4}).{0,12}?(?:wk|week)[ _-]?(\d{1,2})", filename, re.I)
        if m:
            year = year or int(m.group(1))
            week = week or int(m.group(2))
        else:
            m2 = re.search(r"(20\d{2}).{0,12}?(\d{1,2})(?:st|nd|rd|th)?\s*(?:week|wk)", filename, re.I)
            if m2:
                year = year or int(m2.group(1))
                week = week or int(m2.group(2))
    return year, week

MH_UID_LINE_RE = re.compile(
    r"(?P<uid>MH/[A-Z]{2,4}/\d{4}/\d{2}/\d{2,6})\s+"
    r"(?P<state>Maharashtra)\s+"
    r"(?P<district>[A-Za-z().\-\s]+?)\s+"
    r"(?P<disease>[A-Za-z/()&\-\s]+?)\s+"
    r"(?P<cases>\d+(?:/\d+)?)\s+"
    r"(?P<deaths>\d+)\s+"
    r"(?P<start>\d{1,2}-\d{1,2}-\d{2,4})",
    re.I
)

@dataclass
class Row:
    report_year: Optional[int]
    report_week: Optional[int]
    district: str
    disease: str
    category: str
    cases: Optional[int]
    source_pdf: str
    uid: Optional[str]

def _parse_from_tables(tables: List[List[List[str]]]) -> List[Dict[str, Any]]:
    hits = []
    for rows in tables:
        if not rows:
            continue
        # find header row within first few rows
        h_idx = None
        for i, r in enumerate(rows[:6]):
            j = " ".join((_clean_cell(c) or "").lower() for c in r)
            if any(k in j for k in HEADER_KEYS):
                h_idx = i
                break
        if h_idx is None:
            continue

        header = rows[h_idx]
        data   = rows[h_idx+1:]

        hj = [_clean_cell(c).lower() for c in header]
        idx_state = _find_col(hj, ["name of state/ut","name of state-ut"])
        idx_dist  = _find_col(hj, ["name of district"])
        idx_dis   = _find_col(hj, ["disease/ illness","disease/illness","disease- illness"])
        idx_cases = _find_col(hj, ["no. of cases"])
        idx_uid   = _find_col(hj, ["unique id"])

        for r in data:
            c = [_clean_cell(x) for x in r] + [""]*20
            state = c[idx_state] if idx_state is not None else ""
            uid   = c[idx_uid] if idx_uid is not None else ""
            if not (state.lower().startswith("mahar")) and not uid.upper().startswith("MH/"):
                continue
            disease_raw = c[idx_dis] if idx_dis is not None else ""
            disease_clean = strip_leading_enumeration(disease_raw)

            hits.append({
                "uid": uid.strip() or None,
                "district": (c[idx_dist] if idx_dist is not None else "").strip(),
                "disease": disease_clean,
                "cases": _to_int_or_none(c[idx_cases] if idx_cases is not None else None),
            })
    return hits

def _parse_from_text_pages(pages_text: List[str]) -> List[Dict[str, Any]]:
    hits = []
    for t in pages_text:
        compact = re.sub(r"[ \t]*\n[ \t]*", " ", t or "")
        for m in MH_UID_LINE_RE.finditer(compact):
            hits.append({
                "uid": _clean_cell(m.group("uid")),
                "district": _clean_cell(m.group("district")),
                "disease": strip_leading_enumeration(_clean_cell(m.group("disease"))),
                "cases": _to_int_or_none(m.group("cases")),
            })
    return hits

def parse_pdf_maharashtra(path: str) -> List[Row]:
    rows: List[Row] = []
    hits: List[Dict[str, Any]] = []
    header_yw: Tuple[Optional[int], Optional[int]] = (None, None)

    try:
        with suppress_stderr():
            with pdfplumber.open(path) as pdf:
                p0 = pdf.pages[0].extract_text() or ""
                header_yw = parse_header_year_week(p0, os.path.basename(path))

                tables = _collect_tables(pdf)
                hits = _parse_from_tables(tables)

                if not hits:
                    pages_text = []
                    for pg in pdf.pages:
                        try:
                            pages_text.append(pg.extract_text() or "")
                        except Exception:
                            pages_text.append("")
                    hits = _parse_from_text_pages(pages_text)

    except Exception:
        hits = []

    seen = set()
    for rec in hits:
        uid = (rec.get("uid") or "").strip()
        key = uid or (rec.get("district",""), rec.get("disease",""), rec.get("cases",None))
        if key in seen:
            continue
        seen.add(key)

        disease = rec.get("disease","")
        cat = classify_category_max(disease)
        if cat is None:
            continue

        # ✅ FIXED: y,w must be scalars, not tuples
        y, w = None, None
        if uid:
            m = UID_YW_RE.match(uid.strip())
            if m:
                y, w = int(m.group("year")), int(m.group("wk"))
        if (y is None) or (w is None):
            y, w = header_yw

        # if still missing, skip (can't place in time series)
        if y is None or w is None:
            continue

        if not (YEAR_MIN <= y <= YEAR_MAX):
            continue

        rows.append(Row(
            report_year=y,
            report_week=w,
            district=rec.get("district","").strip(),
            disease=disease.strip(),
            category=cat,
            cases=rec.get("cases"),
            source_pdf=os.path.basename(path),
            uid=uid or None
        ))

    return rows

# %% ---------- Orchestration ----------
async def run_pipeline():
    print(f"Discovering Weekly Outbreak Report PDFs (>= {YEAR_MIN} ... {YEAR_MAX})...")
    link_tuples = await crawl_all_links()
    print(f"  Found {len(link_tuples)} candidate links.")

    if not link_tuples:
        print("No links found. (Site layout or network issue)")
        return

    print("Downloading PDFs (skip already-downloaded via manifest)...")
    paths = await download_all(link_tuples, PDF_DIR, MANIFEST_PATH)
    print(f"  PDFs available locally: {len(paths)} in {PDF_DIR}")
    print(f"  Manifest: {MANIFEST_PATH}")

    # Parse all local PDFs
    all_rows: List[Row] = []
    global_seen_uid = set()

    for p in tqdm(paths, desc="Parsing PDFs"):
        recs = parse_pdf_maharashtra(p)
        for r in recs:
            if r.uid and r.uid in global_seen_uid:
                continue
            if r.uid:
                global_seen_uid.add(r.uid)
            all_rows.append(r)

    if not all_rows:
        print("No Maharashtra rows found after parsing & filters.")
        return

    df = pd.DataFrame([r.__dict__ for r in all_rows])

    # Normalize
    df["district"] = df["district"].astype(str).str.replace(r"\s+"," ",regex=True).str.strip()
    df["disease"]  = df["disease"].astype(str).map(strip_leading_enumeration).str.replace(r"\s+"," ",regex=True).str.strip()

    # Strong de-dup
    uid_first = df.dropna(subset=["uid"]).drop_duplicates(subset=["uid"], keep="first")
    no_uid    = df[df["uid"].isna()].drop_duplicates(
        subset=["report_year","report_week","district","disease","cases","source_pdf"], keep="first"
    )
    df_final = pd.concat([uid_first, no_uid], ignore_index=True)

    # Sort + save
    df_final = df_final.sort_values(["report_year","report_week","district","disease","source_pdf"], ignore_index=True)
    df_final.to_csv(OUT_CSV, index=False, encoding="utf-8")

    print(f"\n✅ Wrote {len(df_final)} rows → {OUT_CSV}")

    # Quick health checks
    latest = df_final.sort_values(["report_year","report_week"]).tail(1)[["report_year","report_week","district","disease","cases","uid","source_pdf"]]
    print("\nLatest row in dataset:")
    print(latest.to_string(index=False))

    print("\nRows per year (top):")
    print(df_final.groupby("report_year").size().sort_values(ascending=False).head(10).to_string())

# %% ---------- RUN ----------
try:
    asyncio.run(run_pipeline())
except RuntimeError:
    import nest_asyncio
    nest_asyncio.apply()
    loop = asyncio.get_event_loop()
    loop.create_task(run_pipeline())

Discovering Weekly Outbreak Report PDFs (>= 2010 ... 2026)...
  Found 834 candidate links.


  PDFs available locally: 1499 in ./data/pdf
  Manifest: ./data/download_manifest.json


Parsing PDFs: 100%|██████████| 1499/1499 [49:45<00:00,  1.99s/it] 


✅ Wrote 1356 rows → ./data/csv/idsp_maharashtra_vector_water_air_2010to_2026.csv

Latest row in dataset:
 report_year  report_week                   district disease  cases                 uid                   source_pdf
        2025           40 Chhatrapati Sambhajinag ar  Dengue    NaN MH/CHH/2025/40/1847 31351332071764654053 (1).pdf

Rows per year (top):
report_year
2016    264
2015    182
2014    134
2012    104
2013     88
2011     86
2023     78
2017     67
2021     66
2018     65
